# lumina: research reproduction and extension

The code below is generated directly from the repository packages. Unit tests check implementation behavior, not reproduced paper benchmarks. Runs are illustrative and must be validated before claims of improvement.

In [ ]:
import torch
print("torch:",torch.__version__,"CUDA:",torch.cuda.is_available())
!pip -q install Pillow numpy pytest matplotlib opencv-python-headless

In [ ]:
!mkdir -p lumina tests

In [ ]:
%%writefile lumina/__init__.py
"""Lumina research prototype package."""


In [ ]:
%%writefile lumina/train.py
"""Zero-DCE (CVPR 2020): independent from-scratch algorithm reproduction.
Quickstart: python train.py --dataset synthetic --steps 10 --batch-size 2
Better Colab: python train.py --dataset cifar10 --steps 500 --batch-size 8
"""
import argparse, csv, math, random
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageOps
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader


def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


class DCENet(nn.Module):
    """Seven convolutions, symmetric skip connections, 8 RGB curve maps."""
    def __init__(self, width=32):
        super().__init__()
        self.c1=nn.Conv2d(3,width,3,padding=1)
        self.c2=nn.Conv2d(width,width,3,padding=1)
        self.c3=nn.Conv2d(width,width,3,padding=1)
        self.c4=nn.Conv2d(width,width,3,padding=1)
        self.c5=nn.Conv2d(width*2,width,3,padding=1)
        self.c6=nn.Conv2d(width*2,width,3,padding=1)
        self.c7=nn.Conv2d(width*2,24,3,padding=1)
        for m in self.modules():
            if isinstance(m,nn.Conv2d): nn.init.normal_(m.weight,mean=0,std=0.02); nn.init.constant_(m.bias,0)

    def forward(self,x):
        a1=F.relu(self.c1(x)); a2=F.relu(self.c2(a1)); a3=F.relu(self.c3(a2)); a4=F.relu(self.c4(a3))
        a5=F.relu(self.c5(torch.cat([a3,a4],dim=1)))
        a6=F.relu(self.c6(torch.cat([a2,a5],dim=1)))
        curves=torch.tanh(self.c7(torch.cat([a1,a6],dim=1)))
        y=x
        for a in curves.chunk(8,dim=1):
            # Eq. (3): LE_n = LE_(n-1) + A_n * LE_(n-1) * (1-LE_(n-1)).
            # The original released code uses the algebraically sign-flipped
            # a*(y*y-y); tanh makes the model classes equivalent under a -> -a.
            y=y+a*y*(1-y)
        return torch.clamp(y,0,1),curves


def spatial_loss(x,y):
    """Paper Eq. (4): compare absolute four-neighbor gradients of 4x4 means.

    No cyclic/torch.roll boundaries: cyclic wrapping would connect unrelated
    left/right or top/bottom image edges and invent false spatial neighbors.
    """
    x=F.avg_pool2d(x.mean(1,keepdim=True),4)
    y=F.avg_pool2d(y.mean(1,keepdim=True),4)
    if min(x.shape[-2:]) < 2:
        raise ValueError('spatial_loss requires images at least 8x8')
    # Four directions per interior cell. Use non-wrapped shared valid neighbors.
    terms=[]
    for dim in (-2,-1):
        x0=x.narrow(dim,0,x.shape[dim]-1)
        x1=x.narrow(dim,1,x.shape[dim]-1)
        y0=y.narrow(dim,0,y.shape[dim]-1)
        y1=y.narrow(dim,1,y.shape[dim]-1)
        terms.append(((y1-y0).abs()-(x1-x0).abs()).square().mean())
    return sum(terms)/2


def exposure_loss(y,target=0.6):
    """Paper Eq. (5): absolute 16x16 patch-mean exposure deviation."""
    p=min(16,y.shape[-2],y.shape[-1])
    local=F.avg_pool2d(y.mean(dim=1,keepdim=True),p)
    # Published Zero-DCE Eq. (5) uses absolute exposure deviation.
    # Authors' released training code instead uses a squared deviation.
    return (local-target).abs().mean()


def adaptive_exposure_target(x):
    """EXTENSION: tone-aware per-patch exposure to protect existing highlights.

    Uses only the input image (never a paired target); this is not paper Eq. (5).
    """
    p=min(16,x.shape[-2],x.shape[-1])
    local=F.avg_pool2d(x.mean(1,keepdim=True),p)
    return (0.60-0.35*(local-0.30).relu()).clamp(0.42,0.60)

def color_loss(y):
    """Published Eq. (6): sum of squared differences of channel means.

    Note: the authors' released code instead takes a norm of the squared
    channel differences. This implementation deliberately follows the paper.
    """
    ch=y.mean(dim=(2,3))
    return ((ch[:,0]-ch[:,1]).square() +
            (ch[:,0]-ch[:,2]).square() +
            (ch[:,1]-ch[:,2]).square()).mean()


def curve_tv_loss(curves):
    """Paper Eq. (7): sum_c (|horizontal gradient|+|vertical gradient|)^2.

    Average over eight enhancement maps, images, and spatial positions, while
    summing the three RGB channels as in the mathematical expression. The two
    gradients share an interior pixel, with no cyclic boundary artifacts.
    """
    batch,channels,h,w=curves.shape
    if channels!=24 or min(h,w)<2:
        raise ValueError('Expected 8 RGB curve maps and spatial dimensions >=2')
    a=curves.reshape(batch,8,3,h,w)
    dh=(a[...,1:,:-1]-a[...,:-1,:-1]).abs()
    dw=(a[...,:-1,1:]-a[...,:-1,:-1]).abs()
    return (dh+dw).square().sum(dim=2).mean()

def loss_parts(x,y,curves,ablations=(),extension='none'):
    """Baseline is the four paper losses; optional extension changes exposure."""
    if extension not in ('none','adaptive-exposure'):
        raise ValueError('Unknown extension: '+str(extension))
    exposure_target=adaptive_exposure_target(x) if extension=='adaptive-exposure' else .6
    terms=dict(spatial=spatial_loss(x,y),exposure=exposure_loss(y,exposure_target),
               color=color_loss(y),smooth=curve_tv_loss(curves))
    # Published Eq. (8) coefficients. The authors' public training code
    # has different internal losses and scalings; paper-equation reproduction
    # must not be misrepresented as numeric parity with those released weights.
    weights=dict(spatial=1.,exposure=1.,color=.5,smooth=20.)
    total=sum(weights[k]*v for k,v in terms.items() if k not in ablations)
    return total,terms

def synth_image(seed,sz=96):
    r=random.Random(seed)
    img=Image.new('RGB',(sz,sz),(r.randrange(60,220),r.randrange(60,220),r.randrange(60,220)))
    d=ImageDraw.Draw(img)
    for _ in range(12):
        a=r.randrange(sz);b=r.randrange(sz);c=r.randrange(12,sz//2)
        xy=(a-c,b-c,a+c,b+c);col=tuple(r.randrange(20,255) for i in range(3))
        if r.random()<.5: d.ellipse(xy,fill=col)
        else: d.rectangle(xy,fill=col)
    return img


class LowLightDataset(Dataset):
    """Image folder or CIFAR-10 (unpaired), plus dependency-free procedural smoke-test mode."""
    def __init__(self,mode='synthetic',folder='',size=64,count=1500,seed=7):
        self.mode=mode;self.folder=folder;self.size=size;self.count=count;self.seed=seed
        if mode=='folder':
            self.paths=[p for p in Path(folder).rglob('*') if p.suffix.lower() in ('.png','.jpg','.jpeg','.webp')]
            if not self.paths: raise RuntimeError('No images found in --folder')
        if mode=='cifar10':
            from torchvision.datasets import CIFAR10
            self.ds=CIFAR10(root='./data',train=True,download=True)
    def __len__(self):
        if self.mode=='folder':return len(self.paths)
        if self.mode=='cifar10':return min(self.count,len(self.ds))
        return self.count
    def __getitem__(self,index):
        if self.mode=='folder':
            with Image.open(self.paths[index]) as im: img=im.convert('RGB').copy()
        elif self.mode=='cifar10':img=self.ds[index][0].convert('RGB')
        else:img=synth_image(index+self.seed,size_or_default(self.size))
        img=ImageOps.fit(img,(self.size,self.size),method=Image.Resampling.BICUBIC)
        x=np.asarray(img,dtype=np.float32)/255.0
        # Deterministic darkness for validation and variable on-the-fly exposure in training.
        if self.mode!='folder':
            # Procedural / CIFAR toy benchmark ONLY; real paper mode must preserve
            # the actual low-light image intensities rather than darken them again.
            rng=random.Random(self.seed+int(index)*1009)
            gamma=1.4+1.5*rng.random();scale=0.35+0.4*rng.random()
            x=np.clip((x**gamma)*scale,0.,1.)
        return torch.from_numpy(x.copy()).permute(2,0,1)


def size_or_default(sz):return max(sz,32)


def as_pil(x):
    t=x.detach().cpu().float().clamp(0,1)
    return Image.fromarray((t.permute(1,2,0).numpy()*255).astype('uint8'))


def save_comparison(x,y,path):
    a=as_pil(x);b=as_pil(y);g=Image.fromarray(np.asarray(a)).convert('RGB')
    gamma=np.asarray(g).astype('float32')/255.
    gamma=Image.fromarray(np.uint8(255*np.clip(gamma**0.5,0,1)))
    w,h=a.size;out=Image.new('RGB',(w*3,h+24),'white')
    d=ImageDraw.Draw(out)
    for idx,(im,label) in enumerate([(a,'Input'),(gamma,'Gamma baseline'),(b,'Zero-DCE')]):
        out.paste(im,(idx*w,24));d.text((idx*w+3,5),label,fill='black')
    out.save(path)


def quality_stats(x):
    arr=np.asarray(as_pil(x),dtype=np.float32)/255.
    luminance=arr.mean(2)
    return {'brightness':float(luminance.mean()),'contrast_std':float(luminance.std()),'clipped_high_fraction':float((arr>.98).mean())}


def run(args):
    seed_all(args.seed);device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    out=Path(args.out);out.mkdir(parents=True,exist_ok=True)
    payload=torch.load(args.checkpoint,map_location=device,weights_only=True) if args.checkpoint else None
    effective_width=payload.get('width',args.width) if isinstance(payload,dict) else args.width
    model=DCENet(width=effective_width).to(device)
    if payload is not None:
        model.load_state_dict(payload['model'] if isinstance(payload,dict) and 'model' in payload else payload)
    if args.image:
        with Image.open(args.image) as im: rgb=im.convert('RGB')
        rgb.thumbnail((args.max_side,args.max_side),Image.Resampling.LANCZOS)
        x=torch.tensor(np.asarray(rgb,dtype=np.float32).copy()/255.,device=device).permute(2,0,1).unsqueeze(0)
        with torch.no_grad():y,_=model.eval()(x)
        as_pil(y[0]).save(out/'enhanced.png');save_comparison(x[0],y[0],out/'comparison.png')
        print('Saved',out/'enhanced.png');return
    trainset=LowLightDataset(mode=args.dataset,folder=args.folder,size=args.size,count=args.max_images,seed=args.seed)
    dl=DataLoader(trainset,batch_size=args.batch_size,shuffle=True,num_workers=0,drop_last=False)
    opt=torch.optim.Adam(model.parameters(),lr=args.lr)
    iterator=iter(dl);ablations=set(args.ablate.split(',')) if args.ablate else set()
    if not ablations.issubset({'spatial','exposure','color','smooth'}):
        raise ValueError('Unrecognized ablation term(s): '+str(ablations))
    history=[];model.train()
    for step in range(1,args.steps+1):
        try:x=next(iterator)
        except StopIteration:iterator=iter(dl);x=next(iterator)
        x=x.to(device)
        y,curves=model(x);loss,parts=loss_parts(x,y,curves,ablations,args.extension)
        opt.zero_grad();loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),10.);opt.step()
        history.append([step,float(loss.detach()),*[float(parts[n].detach()) for n in ('spatial','exposure','color','smooth')]])
        if step==1 or step%max(1,args.steps//10)==0:
            print(f'step {step}/{args.steps}: loss={float(loss.detach()):.4f}')
    with (out/'loss.csv').open('w',newline='') as f:
        w=csv.writer(f);w.writerow(['step','loss','spatial','exposure','color','smooth']);w.writerows(history)
    model.eval()
    with torch.no_grad():
        sample=trainset[0].unsqueeze(0).to(device);result,_=model(sample)
    save_comparison(sample[0],result[0],out/'comparison.png')
    stats={k:{'input':v,'enhanced':quality_stats(result[0])[k]} for k,v in quality_stats(sample[0]).items()}
    import json
    (out/'metrics.json').write_text(json.dumps(stats,indent=2))
    torch.save({'model':model.state_dict(),'width':effective_width,'steps':args.steps,'extension':args.extension,
                'seed':args.seed,'dataset':args.dataset,'size':args.size},out/'model.pt')
    print('Saved model, loss.csv, comparison.png, metrics.json to',out,'device',device)


def parse_args(argv=None):
    p=argparse.ArgumentParser()
    p.add_argument('--dataset',choices=['synthetic','cifar10','folder'],default='synthetic')
    p.add_argument('--folder',default='');p.add_argument('--size',type=int,default=64)
    p.add_argument('--steps',type=int,default=400);p.add_argument('--batch-size',type=int,default=8)
    p.add_argument('--max-images',type=int,default=1500);p.add_argument('--lr',type=float,default=1e-4)
    p.add_argument('--width',type=int,default=32);p.add_argument('--seed',type=int,default=42)
    p.add_argument('--out',default='outputs');p.add_argument('--checkpoint',default='')
    p.add_argument('--image',default='');p.add_argument('--max-side',type=int,default=1024);p.add_argument('--ablate',default='',help='comma-separated spatial,exposure,color,smooth')
    p.add_argument('--extension',choices=['none','adaptive-exposure'],default='none',help='Keep none for paper objective; adaptive-exposure is an independently configurable exposure extension')
    return p.parse_args(argv)

if __name__=='__main__':run(parse_args())


In [ ]:
%%writefile lumina/benchmark.py
"""Held-out PAIRED SYNTHETIC benchmark (not an official Zero-DCE or LOL benchmark).

Evaluates image restoration after deterministic simulated low-light degradation.
There is no ground truth for real unpaired photos: PSNR/SSIM here are ONLY
valid against the procedurally generated clean reference images.
"""
import argparse
import csv
import json
from pathlib import Path
import numpy as np
from PIL import Image
import torch
from torch.nn import functional as F
from lumina.train import DCENet, synth_image, as_pil, quality_stats


def degrade(clean, seed):
    """Deterministic low-light corruption, independent of original train.py RNG."""
    rng=np.random.default_rng(seed)
    gamma=float(rng.uniform(1.6,2.5))
    strength=float(rng.uniform(.3,.65))
    return np.clip(clean**gamma*strength,0,1).astype(np.float32)


def gamma_correct(x, exponent=.5):
    return torch.clamp(x,0,1).pow(exponent)


def psnr(a,b):
    mse=F.mse_loss(a,b).item()
    return float(10*np.log10(1/max(mse,1e-12)))


def ssim_global(a,b):
    """Global-channel SSIM-like score: explicitly NOT the standard windowed SSIM."""
    # For true standard SSIM compare via pytorch_msssim or skimage.
    dims=(2,3);ma=a.mean(dim=dims);mb=b.mean(dim=dims)
    va=((a-ma[...,None,None])**2).mean(dim=dims)
    vb=((b-mb[...,None,None])**2).mean(dim=dims)
    cov=((a-ma[...,None,None])*(b-mb[...,None,None])).mean(dim=dims)
    c1=.01**2;c2=.03**2
    return float((((2*ma*mb+c1)*(2*cov+c2))/((ma**2+mb**2+c1)*(va+vb+c2))).mean().item())


def run(args):
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ckpt=torch.load(args.checkpoint,weights_only=True,map_location=device)
    model=DCENet(width=ckpt.get('width',32)).to(device).eval()
    model.load_state_dict(ckpt['model'])
    out=Path(args.out);out.mkdir(parents=True,exist_ok=True)
    rows=[]
    for i in range(args.n):
        # disjoint train/test synthetic seeds: training uses 42+train_idx by default
        seed=100_000+i
        ref=np.asarray(synth_image(seed,args.size),dtype=np.float32)/255.0
        dark=degrade(ref,seed)
        x=torch.from_numpy(dark).permute(2,0,1).unsqueeze(0).to(device)
        target=torch.from_numpy(ref.copy()).permute(2,0,1).unsqueeze(0).to(device)
        with torch.no_grad():
            enhanced=model(x)[0]
            gamma=gamma_correct(x)
        row={'seed':seed}
        for label,result in [('input',x),('gamma',gamma),('zero_dce',enhanced)]:
            row[f'{label}_psnr']=psnr(result,target)
            row[f'{label}_global_ssim']=ssim_global(result,target)
            row[f'{label}_brightness']=quality_stats(result[0])['brightness']
        rows.append(row)
        if i<min(args.preview,4):
            imgs=[as_pil(v[0]) for v in (x,gamma,enhanced,target)]
            canvas=Image.new('RGB',(args.size*4,args.size),'white')
            for j,im in enumerate(imgs):canvas.paste(im,(j*args.size,0))
            canvas.resize((args.size*8,args.size*2)).save(out/f'example_{i:02d}.png')
    keys=list(rows[0])
    with (out/'per_image.csv').open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=keys);writer.writeheader();writer.writerows(rows)
    summary={k:{'mean':float(np.mean([r[k] for r in rows])),
                'std':float(np.std([r[k] for r in rows]))} for k in keys if k!='seed'}
    summary['evaluation_protocol']='Held-out procedurally-generated images, with deterministic simulated low-light corruption; not a real-image or official-paper benchmark.'
    summary['metric_note']='global_ssim is the global-statistics SSIM variant (not windowed SSIM); use external library for standardized SSIM.'
    summary['n_images']=len(rows)
    (out/'summary.json').write_text(json.dumps(summary,indent=2))
    print(json.dumps(summary,indent=2))


def parse_args():
    p=argparse.ArgumentParser()
    p.add_argument('--checkpoint',default='outputs/model.pt')
    p.add_argument('--n',type=int,default=64)
    p.add_argument('--size',type=int,default=64)
    p.add_argument('--preview',type=int,default=4)
    p.add_argument('--out',default='benchmark_outputs')
    return p.parse_args()

if __name__=='__main__':run(parse_args())


In [ ]:
%%writefile lumina/evaluate_advanced.py
"""Paired low-light restoration benchmark with paired bootstrap uncertainty.

For synthetic tests the clean reference is procedural. For --dark-dir/--reference-dir,
input images must be aligned and matched by filename stem. Do not train on the test pairs.
This is an independent evaluation protocol, NOT official Zero-DCE leaderboard parity.
"""
import argparse,csv,json
from pathlib import Path
import numpy as np
import torch
from PIL import Image,ImageOps,ImageDraw
from lumina.train import DCENet,synth_image,as_pil
from lumina.benchmark import degrade,psnr,gamma_correct

SUFFIXES={'.png','.jpeg','.jpg','.webp','.bmp'}

def paired_files(dark_dir,ref_dir):
    def listing(d):
        paths=[p for p in Path(d).rglob('*') if p.suffix.lower() in SUFFIXES]
        byname={}
        for p in paths:
            key=p.stem.lower()
            if key in byname:raise ValueError(f'Duplicate basename: {key} in {d}')
            byname[key]=p
        return byname
    a,b=listing(dark_dir),listing(ref_dir)
    if not a or set(a)!=set(b):
        raise ValueError('Dark/reference folders must contain matching nonempty filename stems')
    return [(key,a[key],b[key]) for key in sorted(a)]


def load_pair(low_path,ref_path,max_side):
    with Image.open(low_path) as f: dark=f.convert('RGB').copy()
    with Image.open(ref_path) as f: ref=f.convert('RGB').copy()
    if dark.size != ref.size:
        raise ValueError(f'Unaligned pair: {low_path} / {ref_path} dimensions {dark.size} != {ref.size}')
    dark.thumbnail((max_side,max_side),Image.Resampling.LANCZOS)
    ref.thumbnail((max_side,max_side),Image.Resampling.LANCZOS)
    x=torch.from_numpy(np.asarray(dark,dtype=np.float32).copy()).permute(2,0,1).unsqueeze(0)/255
    y=torch.from_numpy(np.asarray(ref,dtype=np.float32).copy()).permute(2,0,1).unsqueeze(0)/255
    return x,y


def bootstrap_ci(values,seed=42,repeats=2000):
    values=np.asarray(values,dtype=np.float64)
    if not len(values):raise ValueError('Empty evaluation set')
    rng=np.random.default_rng(seed)
    samples=rng.integers(0,len(values),(repeats,len(values)))
    means=values[samples].mean(axis=1)
    return [float(v) for v in np.quantile(means,[.025,.975])]


def evaluate(checkpoint,out='eval_advanced',dark_dir='',reference_dir='',n=64,size=64,max_side=1024,seed=2026,preview=5):
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    weights=torch.load(checkpoint,map_location=device,weights_only=True)
    model=DCENet(width=int(weights.get('width',32))).to(device).eval()
    model.load_state_dict(weights['model'])
    folder=Path(out);folder.mkdir(parents=True,exist_ok=True)
    if bool(dark_dir)!=bool(reference_dir):raise ValueError('Specify both --dark-dir and --reference-dir')
    use_pairs=bool(dark_dir)
    pairs=paired_files(dark_dir,reference_dir) if use_pairs else list(range(n))
    if not pairs:raise ValueError('No images for evaluation')
    rows=[]
    with torch.inference_mode():
        for i,item in enumerate(pairs):
            if use_pairs:
                name,low,reference=item;x,target=load_pair(low,reference,max_side)
            else:
                name=str(100000+i)
                ref=np.asarray(synth_image(100000+i,size),dtype=np.float32)/255.
                dark=degrade(ref,100000+i)
                x=torch.from_numpy(dark).permute(2,0,1).unsqueeze(0)
                target=torch.from_numpy(ref.copy()).permute(2,0,1).unsqueeze(0)
            x=x.to(device);target=target.to(device)
            generated=model(x)[0]
            gamma=gamma_correct(x)
            mse=lambda a,b:float((a-b).square().mean().item())
            row=dict(image=name,input_psnr=psnr(x,target),gamma_psnr=psnr(gamma,target),model_psnr=psnr(generated,target),
                     input_mae=float((x-target).abs().mean().item()),
                     gamma_mae=float((gamma-target).abs().mean().item()),
                     model_mae=float((generated-target).abs().mean().item()),
                     model_clip_pct=float((generated>.98).float().mean().item()*100),
                     model_minus_gamma_psnr=float(psnr(generated,target)-psnr(gamma,target)))
            rows.append(row)
            if i<preview:
                images=[as_pil(z[0]) for z in (x,gamma,generated,target)]
                w,h=images[0].size;panel=Image.new('RGB',(w*4,h+28),'white')
                draw=ImageDraw.Draw(panel)
                for j,(im,label) in enumerate(zip(images,['LOW-LIGHT','GAMMA','ZERO-DCE','REFERENCE'])):
                    panel.paste(im,(j*w,28));draw.text((j*w+4,5),label,fill='#111111')
                panel.save(folder/f'comparison_{i:02d}.png')
    with (folder/'per_image.csv').open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
    names=['input_psnr','gamma_psnr','model_psnr','input_mae','gamma_mae','model_mae','model_clip_pct','model_minus_gamma_psnr']
    result={'n':len(rows),'dataset_type':'paired-real' if use_pairs else 'held-out-procedural',
            'protocol':'Paired PSNR/MAE for aligned reference images; held-out synthetic unless paired real folders supplied.',
            'means':{k:float(np.mean([row[k] for row in rows])) for k in names},
            'model_minus_gamma_psnr_95pct_bootstrap_ci':bootstrap_ci([r['model_minus_gamma_psnr'] for r in rows],seed=seed),
            'bootstrap_seed':seed,'no_paper_benchmark_claim':True}
    (folder/'summary.json').write_text(json.dumps(result,indent=2));print(json.dumps(result,indent=2))
    return result

if __name__=='__main__':
    ap=argparse.ArgumentParser()
    ap.add_argument('--checkpoint',required=True);ap.add_argument('--out',default='eval_advanced')
    ap.add_argument('--dark-dir',default='');ap.add_argument('--reference-dir',default='')
    ap.add_argument('--n',type=int,default=64);ap.add_argument('--size',type=int,default=64)
    ap.add_argument('--max-side',type=int,default=1024);ap.add_argument('--preview',type=int,default=5)
    ap.add_argument('--seed',type=int,default=2026)
    args=ap.parse_args()
    evaluate(args.checkpoint,args.out,args.dark_dir,args.reference_dir,args.n,args.size,args.max_side,args.seed,args.preview)


In [ ]:
%%writefile lumina/demo.py
"""Creative photo editor: trained Zero-DCE enhancement + reversible strength and white balance.

All manual controls are postprocessing (not learned), clearly separated from the model.
Requires pip install gradio, trained checkpoint.
"""
import argparse
import numpy as np
from PIL import Image
import torch
from lumina.train import DCENet


def edit_image(image, model, device, strength=1., warmth=0., max_side=1400):
    if image is None:return None,None
    original=image.convert('RGB').copy();original.thumbnail((max_side,max_side),Image.Resampling.LANCZOS)
    a=np.asarray(original,dtype=np.float32).copy()/255
    x=torch.from_numpy(a).permute(2,0,1)[None].to(device)
    with torch.inference_mode(): enhanced,_=model(x)
    enhanced=enhanced[0].cpu().permute(1,2,0).numpy()
    # Exposure blend: strength=0 is exactly the source photo.
    output=np.clip(a*(1-float(strength))+enhanced*float(strength),0,1)
    # Manual warm/cool balance, intentionally separate from learned Zero-DCE.
    w=float(warmth)/100
    output=np.clip(output*np.array([1+.22*w,1.,1-.22*w])[None,None,:],0,1)
    return original,Image.fromarray(np.uint8(output*255))


def create_app(checkpoint='outputs/model.pt'):
    import gradio as gr
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    saved=torch.load(checkpoint,map_location=device,weights_only=True)
    model=DCENet(width=saved.get('width',32)).to(device)
    model.load_state_dict(saved['model']);model.eval()
    def enhance(image,strength,warmth):
        return edit_image(image,model,device,strength,warmth)
    return gr.Interface(enhance,
      inputs=[gr.Image(type='pil',label='Low-light source photo'),
              gr.Slider(0,1,value=1,step=.05,label='Zero-DCE enhancement strength'),
              gr.Slider(-100,100,value=0,step=5,label='Manual temperature (cool → warm)')],
      outputs=[gr.Image(type='pil',label='Original'),gr.Image(type='pil',label='Enhanced / downloadable')],
      title='Lumina · Controllable Low-Light Photo Editor',
      description='From-scratch Zero-DCE research prototype. White balance and strength controls are manual postprocessing. Model quality depends on training.')

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--checkpoint',default='outputs/model.pt')
    create_app(p.parse_args().checkpoint).launch(share=False)


In [ ]:
%%writefile pytest.ini
[pytest]
testpaths = tests
pythonpath = .
addopts = --import-mode=importlib


In [ ]:
%%writefile tests/test_lumina_advanced.py
import numpy as np
import torch
from PIL import Image
from lumina.evaluate_advanced import bootstrap_ci,paired_files,load_pair,evaluate
from lumina.train import DCENet

def test_bootstrap_reproducible_and_singleton():
    assert bootstrap_ci([2,2,2])==[2,2]
    assert bootstrap_ci([1,2,3],seed=12)==bootstrap_ci([1,2,3],seed=12)

def test_paired_real_image_protocol(tmp_path):
    d=tmp_path/'dark';r=tmp_path/'ref';d.mkdir();r.mkdir()
    Image.new('RGB',(32,32),(40,20,20)).save(d/'A.png')
    Image.new('RGB',(32,32),(160,140,140)).save(r/'A.png')
    items=paired_files(d,r)
    assert len(items)==1
    low,ref=load_pair(items[0][1],items[0][2],64)
    assert low.shape==ref.shape==(1,3,32,32)
    ckpt=tmp_path/'m.pt'
    torch.save({'model':DCENet(width=8).state_dict(),'width':8},ckpt)
    result=evaluate(str(ckpt),str(tmp_path/'out'),str(d),str(r),preview=1)
    assert result['dataset_type']=='paired-real' and len(result['model_minus_gamma_psnr_95pct_bootstrap_ci'])==2
    assert (tmp_path/'out/comparison_00.png').is_file()

def test_misalignment_rejected(tmp_path):
    a=tmp_path/'a';b=tmp_path/'b';a.mkdir();b.mkdir()
    Image.new('RGB',(32,32)).save(a/'a.png')
    Image.new('RGB',(20,20)).save(b/'a.png')
    k,x,y=paired_files(a,b)[0]
    try:load_pair(x,y,50);assert False
    except ValueError as e:assert 'Unaligned pair' in str(e)


In [ ]:
%%writefile tests/test_lumina_paper_fidelity.py
"""Deterministic algorithm-level checks against paper math; not performance parity."""
import numpy as np
from PIL import Image
import torch
from lumina.train import DCENet, spatial_loss, exposure_loss, color_loss, curve_tv_loss, adaptive_exposure_target, LowLightDataset


def test_curve_identity_and_eight_step_recurrence():
    net=DCENet(width=8)
    for p in net.parameters():
        p.data.zero_()
    x=torch.rand(1,3,32,32)
    result,curves=net(x)
    assert torch.allclose(result,x,atol=1e-6)
    assert curves.shape[1]==24
    net.c7.bias.data.fill_(0.3)
    with torch.no_grad():result,curves=net(x)
    expected=x.clone()
    for param in curves.chunk(8,dim=1):
        expected=expected+param*expected*(1-expected)
    assert torch.allclose(result,expected,atol=1e-6)


def test_spatial_abs_difference_and_no_wraparound():
    # Reverse patch luminance gradient: abs gradients are identical.
    x=torch.zeros(1,3,8,12)
    y=torch.zeros_like(x)
    x[:,:,:,0:4]=.2;x[:,:,:,4:8]=.5;x[:,:,:,8:12]=.8
    y[:,:,:,0:4]=.8;y[:,:,:,4:8]=.5;y[:,:,:,8:12]=.2
    assert spatial_loss(x,y).abs()<1e-7
    # Even with a changed far edge, it must not be compared cyclically to first edge.
    # Explicit expected adjacent gradient losses: (|.5-.2|-|.5-.2|)^2=0,
    # (|.9-.5|-|.8-.5|)^2=.01, averaged over two horizontal edges and
    # a vertical direction of zero -> .0025.
    y=x.clone();y[:,:,:,8:12]=.9
    assert torch.allclose(spatial_loss(x,y),torch.tensor(.0025),atol=1e-6)


def test_nonreference_loss_properties_and_extension():
    gray=torch.full((2,3,32,32),.6)
    assert exposure_loss(gray).item()<1e-6
    assert torch.allclose(color_loss(gray),torch.tensor(0.))
    assert torch.allclose(curve_tv_loss(torch.ones(2,24,32,32)),torch.tensor(0.))
    x=torch.cat([torch.ones(1,3,32,16)*.03,torch.ones(1,3,32,16)*.8],dim=-1)
    exposure=adaptive_exposure_target(x)
    assert exposure.shape==(1,1,2,2)
    assert (exposure[...,0] > exposure[...,1]).all()


def test_real_folder_images_not_darkened_twice(tmp_path):
    clean=np.full((32,32,3),128,dtype=np.uint8)
    Image.fromarray(clean).save(tmp_path/'already_dark.jpg')
    ds=LowLightDataset(mode='folder',folder=str(tmp_path),size=32)
    a=ds[0];b=ds[0]
    assert torch.allclose(a,b)
    assert .48<a.mean()<.52


def test_paper_exposure_absolute_and_color_sum_squares():
    sample=torch.full((1,3,32,32),0.4)
    # Eq. (5): |0.4-0.6|, not the squared-error in authors' public code.
    assert torch.allclose(exposure_loss(sample),torch.tensor(0.2),atol=1e-6)
    color=torch.zeros(1,3,32,32)
    color[:,0]=1.0
    # pairwise differences: (R-G)=1, (R-B)=1, (G-B)=0
    assert torch.allclose(color_loss(color),torch.tensor(2.0),atol=1e-6)


def test_exact_dce_net_parameter_count():
    net=DCENet()
    assert sum(p.numel() for p in net.parameters())==79416


def test_four_losses_have_finite_parameter_gradients():
    from lumina.train import loss_parts
    model=DCENet(width=8)
    x=torch.rand(2,3,32,32)
    y,curves=model(x)
    total,_=loss_parts(x,y,curves)
    total.backward()
    assert torch.isfinite(total)
    assert any(p.grad is not None for p in model.parameters())
    assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)


In [ ]:
%%writefile tests/test_lumina_project.py
import torch
from lumina.train import DCENet,loss_parts,LowLightDataset

def test_forward_and_gradients():
    net=DCENet(width=8);x=torch.rand(2,3,32,32)*.2
    y,a=net(x);assert y.shape==x.shape and a.shape==(2,24,32,32)
    loss,parts=loss_parts(x,y,a);loss.backward()
    assert torch.isfinite(loss) and all(torch.isfinite(v).all() for v in parts.values())
    assert net.c1.weight.grad is not None

def test_dataset():
    x=LowLightDataset('synthetic',size=32,count=4)[0]
    assert x.shape==(3,32,32) and 0<=x.min()<=x.max()<=1


def test_benchmark_degradation_and_metrics():
    from lumina.benchmark import degrade, psnr, ssim_global
    import numpy as np
    ref=np.ones((12,12,3),dtype=np.float32)*.8
    a=degrade(ref,123);b=degrade(ref,123)
    assert np.allclose(a,b) and float(a.mean())<.8
    ref_t=torch.ones(1,3,12,12)*.8
    assert psnr(ref_t,ref_t)>=100
    assert .99<=ssim_global(ref_t,ref_t)<=1.01


In [ ]:
!python -m pytest -q tests/test_lumina_*.py

## Controlled experiments
Each training configuration is a starting point, not evidence of convergence.

In [ ]:
!python -m lumina.train --dataset synthetic --steps 450 --size 64 --batch-size 8 --extension none --seed 42 --out baseline

In [ ]:
!python -m lumina.train --dataset synthetic --steps 450 --size 64 --batch-size 8 --extension adaptive-exposure --seed 42 --out extension

In [ ]:
!python -m lumina.evaluate_advanced --checkpoint baseline/model.pt --n 48 --size 64 --out eval_baseline

In [ ]:
!python -m lumina.evaluate_advanced --checkpoint extension/model.pt --n 48 --size 64 --out eval_extension

## Preserve artifacts
Download before the Colab runtime is recycled.

In [ ]:
from pathlib import Path
from google.colab import files
import shutil
for folder in ('baseline','extension','paired_eval','pipeline_smoke_reward_probe','learned_proxy_smoke','eval_baseline','eval_extension'):
    if Path(folder).is_dir():
        archive=shutil.make_archive(folder,'zip',folder)
        files.download(archive)